# Spatial Data Science with an LoD2 city model

The purpose of this Notebook is to illustrate a CityJSON-based spatial data science workflow with a LoD2 city model.

In [1]:
#load the magic

%matplotlib inline
import os
import sys
from pathlib import Path
import webbrowser

import numpy as np
import pandas as pd
import shapely
from shapely.geometry import Polygon, shape, mapping
import json
import geojson

from osgeo import gdal, osr

import matplotlib.pyplot as plt

In [2]:
import LoD2city3D

from cjio import cityjson

In [3]:
jparams = json.load(open('uEstate_param.json'))   

In [4]:
# With the new 0.10.x syntax:
with open(jparams['cjsn_out'], 'r') as f:
    cm = cityjson.reader(f)

In [5]:
print(cm)

CityJSON version = 2.0.0
EPSG = 32734
bbox = [ 263778.110 6241078.022 28.800 265430.307 6242637.108 193.800 ]
Extensions = ['Energy']
=== CityObjects ===
|-- TINRelief (1)
|-- Road (58)
|-- Building (308)
    |-- BuildingInstallation (145)
        |-- +Energy-PVSystem (145)
|-- LandUse (3)
|-- SolitaryVegetationObject (11)
materials = False
textures = False


In [8]:
# 1. Grab the objects directly from the reader
buildings = {
    obj_id: obj 
    for obj_id, obj in cm.j['CityObjects'].items() 
    if obj.get('type') == 'Building'
}
vertices = cm.j['vertices']

In [9]:

# 2. Call the library function to extract LoD1 and LoD2 data
geojson_output = LoD2city3D.extract_lod_surfaces(buildings, vertices, crs_name="EPSG:32734")

# 3. Flatten the GeoJSON features list directly into a DataFrame
# (Extracting properties and geometry shapes as rows)
rows = []
for feature in geojson_output['features']:
    data_row = feature['properties'].copy()
    data_row['geometry'] = feature['geometry']  # Raw coordinates & type dict
    rows.append(data_row)

gdf = pd.DataFrame(rows)

# Now you have a clean Pandas DataFrame containing all attributes and geometries
gdf.head(2)

,building_id,lod,surface_type,geometry_type,true_3d_area,tilt_deg,aspect_deg,mean_z,attr_osm_id,attr_address,attr_building,attr_building:levels,attr_operator,attr_building_height,attr_roof_height,attr_ground_height,attr_plus_code,attr_footprint,geometry
0,osm_13705114,1.0,GenericSurface,solid,5.362174e+08,90.0,204.054416,43560.0,13705114,197 Nelson Mandela Boulevard Cape Town,warehouse,2.0,Oasis,6.9,77.0,69.6,4FRW3F72+C5V,"[[[264304.86, 6241972.753], [264375.824, 62419...","{'type': 'Polygon', 'coordinates': [[[-299348...."
1,osm_13705114,1.0,GenericSurface,solid,2.017076e+08,90.0,114.281097,43560.0,13705114,197 Nelson Mandela Boulevard Cape Town,warehouse,2.0,Oasis,6.9,77.0,69.6,4FRW3F72+C5V,"[[[264304.86, 6241972.753], [264375.824, 62419...","{'type': 'Polygon', 'coordinates': [[[-228384...."


In [14]:
gdf.loc[1914].geometry

{'type': 'Polygon',
 'coordinates': [[[268820.0, -154106.0],
   [275265.0, -142448.0],
   [277103.0, -143457.0],
   [281574.0, -135377.0],
   [274269.0, -131374.0],
   [270055.0, -139004.0],
   [268414.0, -138112.0],
   [261721.0, -150220.0],
   [268820.0, -154106.0]]]}

In [13]:
gdf.tail(2)

,building_id,lod,surface_type,geometry_type,true_3d_area,tilt_deg,aspect_deg,mean_z,attr_osm_id,attr_address,attr_building,attr_building:levels,attr_operator,attr_building_height,attr_roof_height,attr_ground_height,attr_plus_code,attr_footprint,geometry
1914,osm_1469054027,1.0,GenericSurface,solid,185659314.5,0.0,0.0,56900.0,1469054027,66 Selbourne Road University Estate Cape Town,house,2.0,NaN,6.9,86.3,78.8,4FRW3F64+FCM,"[[[264873.028, 6241703.459], [264879.473, 6241...","{'type': 'Polygon', 'coordinates': [[[268820.0..."
1915,osm_1469054027,1.0,GenericSurface,solid,185659314.5,0.0,0.0,50000.0,1469054027,66 Selbourne Road University Estate Cape Town,house,2.0,NaN,6.9,86.3,78.8,4FRW3F64+FCM,"[[[264873.028, 6241703.459], [264879.473, 6241...","{'type': 'Polygon', 'coordinates': [[[261721.0..."


In [15]:
unique_building_count = gdf['building_id'].nunique()
print(f"Total number of unique buildings: {unique_building_count}")

Total number of unique buildings: 225
